# Bonus: Part 1 across the whole news dataset

Generative AI - Assignment 1
Darrsheni Sapovadia (27PGAI0063)

The bonus run for Part 1: all 2,225 articles instead of the first 30. Same three tasks as the
graded notebook, topic, summary and entities, just over everything.

## Why this one runs locally

Groq is no use for a dataset this size. The free tier allows 200,000 tokens a day and this
needs well over a million, so it runs dry long before the job is done. The brief suggests
Ollama with a small local model instead, which is what this notebook uses. No rate limits,
just a slow laptop.

Three things make the full run finish in a sensible time:

- **One call per row.** The graded notebook used a separate prompt for each task. Here the
  model returns everything in one JSON object, which is a third of the work for Part 1.
- **Eight requests at once.** A 3b model does not come close to using eight cores on its own,
  so running several at a time roughly triples the throughput. Ollama needs to be started
  with `OLLAMA_NUM_PARALLEL` set to match.
- **A JSON schema rather than just asking for JSON.** This one mattered more than I expected.
  Asking politely for one of five categories, a 3b model happily answers "Art" or "Sports"
  instead, and my first attempt was getting about a third of them right. Passing a schema
  with an `enum` makes Ollama constrain the output as it generates, so an invalid category
  is not possible any more.

It is still a much smaller model than the graded notebooks use, and it is less accurate. The
brief does ask to balance inference speed against accuracy, and a 3b model at roughly ten
seconds a row is where that lands on this hardware.

## Setup

In [ ]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL = "llama3.2:3b"
WORKERS = 8

print("Model:", MODEL, "| parallel requests:", WORKERS)

In [ ]:
def find_json(text):
    """Grab the JSON object out of a reply and ignore anything around it."""
    found = re.search(r"\{.*\}", text, re.S)
    if not found:
        return {}
    try:
        return json.loads(found.group())
    except json.JSONDecodeError:
        return {}


def run_all(work, handle, checkpoint, every=100):
    """Run `handle` over every row, saving progress as it goes.

    This takes hours on a laptop CPU, so it writes a checkpoint every hundred
    rows. If the run dies partway through, running the cell again picks up from
    where it stopped rather than starting the whole thing over.
    """
    done = {}
    if os.path.exists(checkpoint):
        saved = pd.read_csv(checkpoint).fillna("")
        done = {int(r["row"]): r.to_dict() for _, r in saved.iterrows()}
        print(f"found a checkpoint with {len(done)} rows already done")

    todo = [w for w in work if w[0] not in done]
    print(f"{len(todo)} rows still to do")

    started_with = len(done)
    start = time.time()

    for at in range(0, len(todo), every):
        chunk = todo[at:at + every]
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for result in pool.map(handle, chunk):
                done[result["row"]] = result

        pd.DataFrame(sorted(done.values(), key=lambda r: r["row"])).to_csv(
            checkpoint, index=False
        )
        per_row = (time.time() - start) / max(1, len(done) - started_with)
        left = (len(work) - len(done)) * per_row / 60
        print(f"  {len(done)}/{len(work)} done, roughly {left:.0f} min left")

    return pd.DataFrame(sorted(done.values(), key=lambda r: r["row"]))

## Load the whole dataset

In [ ]:
news = pd.read_csv("../data/bbc-news-data.csv", sep="\t").reset_index(drop=True)

print("Articles to process:", len(news))
news.head(3)

## The prompt and the schema

The wording is the same as the graded notebook, including the rule about business and the
economy that I worked out there, only now it asks for all three answers at once. The schema
underneath it is what forces `topic` to be one of the five real categories.

In [ ]:
CATEGORIES = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

SCHEMA = {
    "type": "object",
    "properties": {
        "topic": {"type": "string", "enum": CATEGORIES},
        "summary": {"type": "string"},
        "entities": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["topic", "summary", "entities"],
}

llm = ChatOllama(model=MODEL, temperature=0, num_predict=400, format=SCHEMA)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a news desk editor. You analyse articles and reply with JSON only."),
    ("human",
     "Classify this article, summarise it, and list its key entities.\n\n"
     "topic must be exactly one of Business, Entertainment, Politics, Sport, Tech. "
     "Anything about companies, markets, jobs, trade, prices or the economy is Business, "
     "even when a government or regulator is involved. Keep Politics for elections, parties "
     "and the running of government. Arts, film, music, celebrity and TV are Entertainment. "
     "Science, computing, internet and gadgets are Tech.\n"
     "summary must be 2 to 3 sentences, covering only what the article says.\n"
     "entities are the important people, organisations and places.\n\n"
     "Article:\n{article}"),
])

chain = prompt | llm | StrOutputParser()

In [ ]:
def analyse(item):
    """Do all three tasks for one article and return a row of results."""
    position, title, article = item
    topic, summary, entities = "Other", "", ""

    try:
        data = find_json(chain.invoke({"article": article[:1000]}))

        raw = str(data.get("topic", ""))
        for category in CATEGORIES:
            if category.lower() in raw.lower():
                topic = category
                break

        summary = str(data.get("summary", "")).strip()

        found = data.get("entities", [])
        if isinstance(found, str):
            found = [found]
        entities = ", ".join(dict.fromkeys(str(e).strip() for e in found if str(e).strip()))
    except Exception as error:
        # one bad row should not bring down a run that has been going for hours
        summary = f"failed: {type(error).__name__}"

    return {
        "row": position,
        "Detected_Topic": topic,
        "Summary": summary,
        "Key_Entities": entities,
    }

## Run it

The long part. Roughly ten seconds an article, so about six hours for all 2,225. Progress
prints every hundred rows and the checkpoint sits next to it in `outputs/`.

In [ ]:
work = [(i, r["title"], r["content"]) for i, r in news.iterrows()]

results = run_all(work, analyse, "../outputs/bonus_part1_checkpoint.csv")

print("\nFinished:", len(results), "articles")

## Put the results back on the dataframe

In [ ]:
full = news.rename(columns={
    "title": "Title",
    "content": "Article_Text",
    "category": "True_Category",
}).copy()
full.insert(0, "Article_ID", full.index + 1)

merged = full.join(results.set_index("row")[["Detected_Topic", "Summary", "Key_Entities"]])
merged = merged[["Article_ID", "Title", "Article_Text", "True_Category",
                 "Detected_Topic", "Summary", "Key_Entities"]]

pd.set_option("display.max_colwidth", 55)
merged.head(15)

In [ ]:
print("Rows:", len(merged))
print("Rows with no summary:", int(merged["Summary"].fillna("").str.len().lt(10).sum()))
print()
print("Topics the model picked:")
print(merged["Detected_Topic"].value_counts().to_string())

## How well did it do

Every article here comes with its real category, so unlike the 30 row notebook this is a
proper test: all five categories, 2,225 rows, nothing cherry picked.

In [ ]:
merged["Match"] = (merged["True_Category"].str.lower()
                   == merged["Detected_Topic"].str.lower())

print(f"Correct: {merged['Match'].sum()} of {len(merged)} "
      f"({100 * merged['Match'].mean():.1f}%)")
print()
print("By real category:")
print(merged.groupby("True_Category")["Match"].agg(["sum", "count"]).to_string())

## Save

In [ ]:
out = merged.drop(columns=["Match"])
out.to_csv("../outputs/bonus_part1_full_news_results.csv", index=False)

print("Saved", len(out), "rows to outputs/bonus_part1_full_news_results.csv")